# Gaussian plume inversion lab
This synthetic lab estimates a fixed-source emission rate from an already retrieved column-enhancement field. It is not a satellite retrieval or free source-localisation exercise.

In [ ]:
import sys
sys.path.insert(0, '../implementations/python/methane-emissions')
import numpy as np
import matplotlib.pyplot as plt
from gaussian_plume_inversion import gaussian_column_kernel, invert_gaussian_plume
x, y = np.meshgrid(np.linspace(-500, 2500, 101), np.linspace(-900, 900, 61))
kernel = gaussian_column_kernel(x, y, 0, 0, 5.5, 30, 0.065)
true_q, true_background = 95.0, 0.0015
rng = np.random.default_rng(23)
observed = true_q * kernel + true_background + rng.normal(0, 2e-6, x.shape)
fit = invert_gaussian_plume(observed, x, y, source_x=0, source_y=0, wind_speed_m_s=5.5, sigma_y0_m=30, dispersion_growth=0.065)
print('True Q:', true_q, 'kg/s; estimated Q:', fit['emission_rate_kg_s'], 'kg/s')
print('True background:', true_background, 'kg/m²; estimated:', fit['background_kg_m2'], 'kg/m²')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for ax, values, title in zip(axes, [observed, fit['fitted_column_kg_m2'], fit['residual_kg_m2']], ['Synthetic observation', 'Fitted column', 'Residual']):
    im = ax.imshow(values, origin='lower', extent=[x.min(), x.max(), y.min(), y.max()], aspect='auto')
    ax.set_title(title); ax.set_xlabel('Downwind x (m)'); ax.set_ylabel('Crosswind y (m)'); fig.colorbar(im, ax=ax)
plt.show()
print('RMS residual (kg/m²):', fit['rms_residual_kg_m2'])


## Interpretation limits
The fit assumes known source location, wind aligned with x, known wind speed, a fixed lateral-dispersion law and a constant background. It is vertically integrated and steady-state. Residual structure, wind uncertainty, source-location uncertainty, retrieval error and model mismatch are not represented in the parameter fit. A numerical recovery on synthetic data validates this implementation contract, not field accuracy.